# 01 · Dados

**Recomendação de notícias com MLOps no Databricks** — FIAP Pós-Tech, Fase 5.

Este notebook monta os dados que o modelo vai usar:

1. **Matérias reais** do sitemap de notícias do O Globo → tabela `bronze_articles`
2. **Cliques sintéticos** de 300 leitores fictícios → tabelas `silver_users` e `silver_clicks`

| Conceito | No Databricks |
|---|---|
| Onde ficam os dados | **Unity Catalog**: `catálogo.schema.tabela` (aqui `workspace.news_recsys_dev`) |
| Arquivos brutos | **Volume**: `/Volumes/workspace/news_recsys_dev/raw/` |
| Tabelas | **Delta Lake** (Parquet com histórico) |

In [ ]:
# Parâmetro do Job (rodando à mão, fica o valor padrão)
dbutils.widgets.text("schema", "news_recsys_dev")
SCHEMA = f"workspace.{dbutils.widgets.get('schema')}"
print(SCHEMA)

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SCHEMA}.raw")

## 1. Matérias reais (sitemap do O Globo)

O GitHub Actions baixa o sitemap todo dia e grava o XML no Volume (o Databricks Free
Edition não tem internet de saída). Aqui lemos esses XMLs. A **editoria** sai da URL:
`oglobo.globo.com/economia/...` → `economia`.

In [ ]:
import xml.etree.ElementTree as ET

import pandas as pd

NS = {
    "sm": "http://www.sitemaps.org/schemas/sitemap/0.9",
    "news": "http://www.google.com/schemas/sitemap-news/0.9",
    "image": "http://www.google.com/schemas/sitemap-image/1.1",
}


def parse_sitemap(xml_bytes):
    linhas = []
    for url in ET.fromstring(xml_bytes).findall("sm:url", NS):
        linhas.append({
            "url": url.findtext("sm:loc", "", NS).strip(),
            "title": url.findtext("news:news/news:title", "", NS).strip(),
            "published_at": url.findtext("news:news/news:publication_date", "", NS),
            "image_url": url.findtext("image:image/image:loc", "", NS),
        })
    return pd.DataFrame(linhas)


def editoria(url):
    secao = url.split("/")[3]
    return {"apuracao-eleicoes-2026": "politica", "play": "cultura"}.get(secao, secao)

In [ ]:
from pathlib import Path

arquivos = sorted(Path(f"/Volumes/{SCHEMA.replace('.', '/')}/raw/sitemaps").glob("*.xml"))
print(len(arquivos), "arquivo(s) de sitemap no Volume")

materias = pd.concat([parse_sitemap(f.read_bytes()) for f in arquivos])
materias["section"] = materias["url"].map(editoria)
materias = materias[(materias["title"] != "") & (materias["section"] != "patrocinado")]
materias = materias.drop_duplicates("url").reset_index(drop=True)

spark.createDataFrame(materias).write.mode("overwrite").saveAsTable(f"{SCHEMA}.bronze_articles")
print(len(materias), "matérias em", f"{SCHEMA}.bronze_articles")
materias["section"].value_counts().head(10)

## 2. Cliques sintéticos

Não temos o histórico de leitura do O Globo (é dado privado). Então **simulamos**
300 leitores, cada um com uma *persona* (torcedor, investidor, politizado...) que
prefere certas editorias. Também há matérias que "viralizam" e notícia velha perde
interesse.

> Em produção, esta tabela viria do analytics do site. **O resto do pipeline não muda.**

In [ ]:
import numpy as np

PERSONAS = {
    "politizado": {"politica": 0.6, "brasil": 0.2, "blogs": 0.2},
    "investidor": {"economia": 0.7, "mundo": 0.2, "politica": 0.1},
    "torcedor": {"esportes": 0.8, "rio": 0.2},
    "carioca": {"rio": 0.6, "cultura": 0.2, "esportes": 0.2},
    "cultural": {"cultura": 0.6, "ela": 0.4},
    "globalista": {"mundo": 0.7, "economia": 0.3},
    "bem-estar": {"saude": 0.6, "ela": 0.4},
    "generalista": {},
}


def simular_cliques(materias, n_usuarios=300, cliques_por_usuario=18, seed=42):
    rng = np.random.default_rng(seed)
    publicada = pd.to_datetime(materias["published_at"], utc=True, format="mixed")
    horas = ((publicada - publicada.min()).dt.total_seconds() / 3600).to_numpy()
    popularidade = rng.lognormal(0, 1, len(materias))  # algumas matérias "viralizam"
    usuarios, cliques = [], []
    for i in range(n_usuarios):
        persona = list(PERSONAS)[i % len(PERSONAS)]
        gosto = materias["section"].map(PERSONAS[persona]).fillna(0.02).to_numpy()
        usuarios.append({"user_id": f"u_{i:03d}", "persona": persona})
        for t in np.sort(rng.uniform(0, horas.max() + 6, cliques_por_usuario)):
            idade = t - horas  # horas desde a publicação (negativo = ainda não saiu)
            peso = gosto**3 * popularidade * np.where(idade >= 0, 0.5 ** (idade / 12), 0)
            j = rng.choice(len(materias), p=peso / peso.sum())
            cliques.append({
                "user_id": f"u_{i:03d}",
                "url": materias["url"].iat[j],
                "event_ts": publicada.min() + pd.Timedelta(hours=float(t)),
            })
    cliques = pd.DataFrame(cliques).drop_duplicates(["user_id", "url"])
    cliques["event_ts"] = cliques["event_ts"].dt.tz_convert(None)
    return pd.DataFrame(usuarios), cliques

In [ ]:
usuarios, cliques = simular_cliques(materias)

spark.createDataFrame(usuarios).write.mode("overwrite").saveAsTable(f"{SCHEMA}.silver_users")
spark.createDataFrame(cliques).write.mode("overwrite").saveAsTable(f"{SCHEMA}.silver_clicks")
print(len(usuarios), "usuários ·", len(cliques), "cliques")
cliques.head()

### Cada persona lê coisas diferentes?

In [ ]:
lidas = cliques.merge(usuarios, on="user_id").merge(materias[["url", "section"]], on="url")
mix = pd.crosstab(lidas["persona"], lidas["section"], normalize="index")
mix[mix.max().nlargest(8).index].round(2).style.background_gradient(cmap="Blues", axis=None)

### O que percebemos?

- Os dados ganharam **endereço no catálogo** (`workspace.news_recsys_dev.*`).
- Cada persona concentra a leitura em poucas editorias: é esse padrão que o modelo
  precisa aprender.
- A simulação usa `seed=42`: rodar de novo gera os mesmos cliques.

**Próximo:** `02_treino`.